# Skipped survey questions: age, income, phone hours

> **Before you run:** choose **Copy to Drive** at the top of this page (or **File → Save a copy in Drive**). Your copy opens in a new tab. In your copy, choose **Runtime → Run all**. Your copy is yours, so Colab runs it without a warning, and it keeps your changes.

**Missing Value Check** · a worked example from the StatNex teaching guide

**The story.** A market-research firm collected 51 short survey answers: age, monthly income and hours spent on the phone per day. Some people skipped questions; the blanks and the "-" entries both mean "no answer".

**The question.** How much is missing, and which question do people avoid?

If you run this page without a copy, Colab first warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `12_utility/missing_value_check.csv` (51 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
No.,Age,Monthly income ($),Phone hours per day
1.0,23.0,1310.0,6.9
2.0,29.0,3000.0,0.9
3.0,28.0,,2.0
4.0,18.0,800.0,5.8
5.0,18.0,3400.0,1.9
6.0,35.0,2120.0,2.8
7.0,,2830.0,4.4
8.0,29.0,3540.0,5.1
9.0,34.0,3340.0,1.6
10.0,27.0,,4.2
11.0,37.0,3820.0,2.2
12.0,48.0,3950.0,5.2
13.0,48.0,4150.0,5.3
14.0,20.0,3110.0,5.4
15.0,30.0,,4.0
16.0,20.0,3260.0,3.9
17.0,44.0,4090.0,4.5
18.0,41.0,2300.0,
19.0,29.0,3630.0,3.9
20.0,49.0,3260.0,2.8
21.0,36.0,2630.0,6.2
22.0,28.0,,7.9
23.0,21.0,800.0,3.8
24.0,44.0,1790.0,3.5
25.0,18.0,800.0,3.9
26.0,18.0,2450.0,5.3
27.0,31.0,2400.0,3.6
28.0,40.0,,8.5
29.0,38.0,1990.0,4.3
30.0,34.0,3640.0,7.8
31.0,,4160.0,5.2
32.0,25.0,2940.0,5.9
33.0,53.0,3590.0,5.2
34.0,42.0,,0.9
35.0,18.0,1940.0,3.2
36.0,44.0,3760.0,4.6
37.0,37.0,3780.0,7.1
38.0,44.0,3680.0,2.8
39.0,29.0,,4.7
40.0,44.0,3220.0,4.4
41.0,18.0,3730.0,
42.0,34.0,3310.0,6.0
43.0,31.0,910.0,1.0
44.0,44.0,4110.0,5.5
45.0,18.0,,3.7
46.0,49.0,2400.0,2.5
47.0,36.0,3980.0,
48.0,38.0,3210.0,3.6
49.0,36.0,2900.0,0.5
50.0,35.0,,4.3
51.0,19.0,2450.0,4.5

## Libraries

In [ ]:
import math

import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

## The data

In [ ]:
# Every cell is sorted into one of four kinds: a number, a blank, text that is not a
# number (such as "unknown"), or a spreadsheet error value (such as #DIV/0!). They are
# counted apart because each has a different cause and fix. Markers such as "-" or
# "NA" count as blanks or as text depending on how the app's import was told to read
# missing values; whatever it read as a blank arrives in data.csv as a blank.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
variables = ["Age", "Monthly income ($)", "Phone hours per day"]
cells = data[variables]

ERROR_VALUES = {"#N/A", "#VALUE!", "#REF!", "#DIV/0!", "#NUM!", "#NAME?", "#NULL!"}


def kind_of(value):
    """Which of the four kinds one cell is."""
    if pd.isna(value):
        return "blank"
    text = str(value).strip()
    if text == "":
        return "blank"
    if text.upper() in ERROR_VALUES:
        return "error"
    # Counted the way an analysis will read the cell: "nan" and "inf" are not
    # usable numbers, so they count as text.
    try:
        number = float(text)
    except ValueError:
        return "text"
    if math.isnan(number) or math.isinf(number):
        return "text"
    return "number"


kinds = cells.map(kind_of)
missing = kinds != "number"
cases = len(cells)

## Overview

In [ ]:
# A case (row) is complete when every variable holds a number. Only complete cases
# survive the listwise deletion most multivariate methods apply. Dropping incomplete
# rows is harmless only if the gaps are missing completely at random (MCAR): whether
# a cell is blank has nothing to do with what the answer would have been. If, say,
# high earners skip the income question, the complete cases are no longer a fair
# sample. These counts show how much is missing, never why.
complete = ~missing.any(axis=1)
has_blank = (kinds == "blank").any(axis=1)
has_text = (kinds == "text").any(axis=1)
has_error = (kinds == "error").any(axis=1)
overview = pd.DataFrame(
    [[len(variables), np.nan],
     [cases, 1.0],
     [complete.sum(), complete.mean()],
     [has_blank.sum(), has_blank.mean()],
     [has_text.sum(), has_text.mean()],
     [has_error.sum(), has_error.mean()]],
    index=["variables (columns)", "cases (rows)", "cases with no missing value",
           "cases with a blank", "cases with text", "cases with an error value"],
    columns=["count", "share"],
)
print(overview)

## Missing values by variable

In [ ]:
by_variable = pd.DataFrame(
    {"valid": (kinds == "number").sum(),
     "missing": missing.sum(),
     "missing rate": missing.sum() / cases,
     "blank": (kinds == "blank").sum(),
     "text": (kinds == "text").sum(),
     "error": (kinds == "error").sum()},
    index=variables,
)
print()
print(by_variable)

## Missing-value patterns

In [ ]:
# Each case is described by the set of variables it is missing; equal sets are
# counted together. Most frequent first; ties in the order of the variable names.
# Rows missing several variables at once, or one question that dominates, point to
# where the gaps come from.
counts = {}
for flags in missing.itertuples(index=False):
    gone = tuple(name for name, is_missing in zip(variables, flags) if is_missing)
    counts[gone] = counts.get(gone, 0) + 1
ordered = sorted(counts.items(), key=lambda item: (-item[1], item[0]))
patterns = pd.DataFrame(
    [[", ".join(gone) if gone else "(nothing missing)", len(gone), count, count / cases]
     for gone, count in ordered],
    columns=["variables missing", "number missing", "cases", "share"],
)
print()
print(patterns.to_string(index=False))

## What the result says

Of the 51 cases (people), 37 have no missing values, a proportion of 0.725. The other 14 each skipped a question.

The missing rate, the share of people with no answer, differs by question. Monthly income has 9 missing answers, a rate of 0.176, or about one person in six. Phone hours has 3 missing (0.059) and age has 2 (0.039). The missingness patterns show that nobody skipped more than one question.

So income is the question people avoid: at least three times as many skipped it as any other question. The counts cannot tell us why. If people with high or low incomes skip more often, the average of the 42 valid incomes will be off, and that decides whether it can be trusted.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.